# Aula 15 — Exercício Prático: classificando diabetes com quatro modelos

**Especialização em Ciência de Dados · Análise e Ciência de Dados · IFRN**

**Exercício para treino — não precisa ser entregue.** A avaliação desta aula é o formulário avaliativo no Moodle.

### O caso
O dataset *Pima Indians Diabetes* (também do **Kaggle**: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database, arquivo `diabetes.csv`) traz **768 mulheres** de origem Pima, com 21 anos ou mais, e o resultado de um teste de diabetes (`Outcome`: 1 = diabetes). Pergunta: **a partir de medidas clínicas simples, os quatro modelos da aula conseguem identificar quem tem diabetes?**

Ao contrário do laboratório (tumores), aqui as classes **se sobrepõem bastante** — você vai ver como isso muda as métricas e a escolha do modelo.

> Fonte original: National Institute of Diabetes and Digestive and Kidney Diseases (EUA), via UCI Machine Learning. Registros anônimos. Exercício de aprendizado de máquina, não ferramenta de diagnóstico.

| Coluna | Descrição |
|---|---|
| `Pregnancies` | número de gestações |
| `Glucose` | glicose plasmática 2 h após teste oral de tolerância (mg/dL) |
| `BloodPressure` | pressão diastólica (mmHg) |
| `SkinThickness` | espessura da dobra cutânea do tríceps (mm) |
| `Insulin` | insulina sérica 2 h (μU/mL) |
| `BMI` | índice de massa corporal (kg/m²) |
| `DiabetesPedigreeFunction` | índice de histórico familiar de diabetes |
| `Age` | idade (anos) |
| **`Outcome`** | **alvo**: 1 = diabetes, 0 = não |

**Como usar:** em cada passo, complete as células marcadas com `# TODO` e confira com o **resultado esperado**. Use `random_state=42` sempre que houver sorteio — assim seus números batem com os esperados. O exercício é para **treino** e **não precisa ser entregue**; o gabarito comentado está em `Aula15_Exercicio_Gabarito.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid", palette="colorblind")
SEMENTE = 42
CORES = {"Regressão logística": "#0173b2", "Árvore de decisão": "#de8f05",
         "Random forest": "#029e73", "KNN": "#cc78bc"}

## Passo 1 — Ler os dados e conhecer o alvo
Leia `dados/diabetes.csv`, mostre o formato e a proporção de cada classe de `Outcome`.

**Resultado esperado:** `(768, 9)`; 500 sem diabetes e 268 com diabetes → **34,9%** de positivos.

In [ ]:
df = ...  # TODO: ler o CSV
# TODO: mostrar df.shape e a proporção de Outcome (value_counts com normalize=True)

## Passo 2 — Zeros que são, na verdade, valores faltantes
Glicose, pressão, dobra cutânea, insulina e IMC iguais a **0** são impossíveis: são medidas que não foram feitas. Conte os zeros dessas cinco colunas e troque-os por `NaN` (a imputação será feita **dentro do pipeline**, para não vazar informação do teste).

**Resultado esperado:** Glucose 5 · BloodPressure 35 · SkinThickness 227 · Insulin 374 · BMI 11.

In [ ]:
colunas_zero = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
# TODO: contar os zeros de cada coluna
# TODO: substituir 0 por np.nan nessas colunas

## Passo 3 — X, y e divisão treino/teste
Separe `X` (8 atributos) e `y` (`Outcome`). Divida com `test_size=0.25`, `stratify=y` e `random_state=SEMENTE`.

**Resultado esperado:** treino `(576, 8)`, teste `(192, 8)`; 34,9% de positivos nos dois (67 no teste).

In [ ]:
X = ...  # TODO
y = ...  # TODO
X_treino, X_teste, y_treino, y_teste = ...  # TODO: train_test_split

## Passo 4 — Baseline
Treine um `DummyClassifier(strategy="most_frequent")` e calcule acurácia e recall no teste.

**Resultado esperado:** acurácia **0,651**, recall **0,000**.

In [ ]:
# TODO: DummyClassifier, fit no treino, acurácia e recall no teste

## Passo 5 — Os quatro pipelines
Monte um dicionário `modelos` com os quatro classificadores. Todos começam com `SimpleImputer(strategy="median")`; **regressão logística e KNN** recebem também `StandardScaler`. Use:
* `LogisticRegression(max_iter=1000)`
* `DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=SEMENTE)`
* `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=SEMENTE)`
* `KNeighborsClassifier(n_neighbors=15)`

**Resultado esperado:** um dicionário com 4 pipelines (nada é impresso além dele).

In [ ]:
modelos = {
    "Regressão logística": ...,  # TODO: Pipeline([("imputar", ...), ("padronizar", ...), ("classificador", ...)])
    "Árvore de decisão": ...,    # TODO (sem padronizar)
    "Random forest": ...,        # TODO (sem padronizar)
    "KNN": ...,                  # TODO
}

## Passo 6 — Comparação com validação cruzada
Use `StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)` e `cross_validate` com `scoring=["accuracy", "recall", "f1", "roc_auc"]`. Monte uma tabela com a média de cada métrica e o desvio-padrão da ROC-AUC.

**Resultado esperado (médias):**

| modelo | acurácia | recall | F1 | ROC-AUC |
|---|---|---|---|---|
| Regressão logística | 0,778 | 0,583 | 0,646 | 0,831 ± 0,023 |
| Árvore de decisão | 0,724 | 0,662 | 0,622 | 0,777 ± 0,050 |
| Random forest | 0,748 | 0,523 | 0,591 | 0,825 ± 0,021 |
| KNN | 0,778 | 0,587 | 0,646 | 0,831 ± 0,032 |

In [ ]:
cv = ...  # TODO
linhas = []
for nome, modelo in modelos.items():
    r = ...  # TODO: cross_validate
    # TODO: guardar as médias em linhas.append({...})
# TODO: pd.DataFrame(linhas)

## Passo 7 — Matrizes de confusão no teste
Treine os quatro modelos no treino completo e desenhe as quatro matrizes de confusão do teste lado a lado.

**Resultado esperado (VN, FP, FN, VP):** regressão logística (105, 20, 33, 34) · árvore (103, 22, 25, 42) · random forest (107, 18, 29, 38) · KNN (104, 21, 32, 35).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.9))
for ax, (nome, modelo) in zip(axes, modelos.items()):
    pass  # TODO: fit, predict e ConfusionMatrixDisplay.from_predictions(..., ax=ax)
plt.show()

## Passo 8 — Curvas ROC
Desenhe as quatro curvas ROC do teste no mesmo gráfico, com a AUC na legenda.

**Resultado esperado (AUC no teste):** regressão logística 0,824 · árvore 0,781 · random forest 0,813 · KNN 0,800.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for nome, modelo in modelos.items():
    pass  # TODO: predict_proba, roc_curve, ax.plot(...)
ax.plot([0, 1], [0, 1], "--", color="gray")
plt.show()

## Passo 9 — Limiar para encontrar mais casos
Com a regressão logística, calcule recall, precisão, FN e FP para os limiares 0,2 · 0,3 · 0,4 · 0,5. Qual é o **maior** limiar com recall ≥ 0,75?

**Resultado esperado:** em 0,5: recall 0,507 (33 FN, 20 FP); em **0,3**: recall **0,776**, precisão 0,571 (15 FN, 39 FP); em 0,2: recall 0,896 (7 FN, 47 FP). Resposta: **0,3**.

In [ ]:
proba = ...  # TODO: probabilidades da regressão logística no teste
# TODO: laço pelos limiares montando uma tabela

## Passo 10 — Conclusão (escreva com suas palavras)
Responda em 3 a 5 linhas: (a) qual modelo você escolheria e por quê; (b) por que as métricas são bem menores que as do laboratório dos tumores; (c) que limiar você usaria numa triagem e qual o custo dessa escolha.

*Sua resposta aqui.*